<a href="https://colab.research.google.com/github/lloydakresi/ml_journey/blob/main/Attention_Mechanisms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%cd drive/MyDrive

/content/drive/MyDrive


In [3]:
import torch
import math
from seq2seq_models import preprocess_text, tokenize_data, pad_or_truncate, Vocab, encode, Embedding
from sequential_models import Linear, Tanh, BasicRNN, LSTMCell, GRUCell

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Text preprocessing

In [5]:
dataset_path = "fra.txt"
src, tgt = [], []

with open(dataset_path) as file_object:
  for i, line in enumerate(file_object):
    result = preprocess_text(line)
    tokenize_data(result, src, tgt)

src = [pad_or_truncate(s) for s in src]
tgt = [pad_or_truncate(t) for t in tgt]
tgt = [["<bos>"] + t for t in tgt]
eng_vocab = Vocab(src)
fr_vocab = Vocab(tgt)

lookup_fxn = lambda sentence, vocab: [encode(vocab, s) for s in sentence]
src_lookup = [lookup_fxn(s, eng_vocab) for s in src]
tgt_lookup = [lookup_fxn(t, fr_vocab) for t in tgt]

src_lookup = torch.tensor(src_lookup, dtype=torch.int32)
tgt_lookup = torch.tensor(tgt_lookup, dtype=torch.int32)

target_label = tgt_lookup[:, 1:]
decoder_input = tgt_lookup[:, :-1]

label_valid_len = (target_label != fr_vocab["<pad>"]).type(torch.int32).sum(1)
decoder_valid_len = (decoder_input != fr_vocab["<pad>"]).type(torch.int32).sum(1)
src_valid_len = (src_lookup != eng_vocab["<pad>"]).type(torch.int32).sum(1)

In [6]:
def masked_softmax(X, valid_lens):
  """
  Mask <pad> tokens
  """
  def _mask(X, valid_len, value=0):
    maxlen = X.shape[-1]
    mask = torch.arange(maxlen) < valid_len.unsqueeze(-1)
    X[~mask] = value
    return X

  if valid_lens is None:
    return torch.nn.functional.softmax(X, dim=-1)
  else:
    if valid_lens.dim() == 1:
      valid_lens = torch.repeat_interleave(valid_lens, X.shape[1])
    else:
      valid_lens = valid_lens.reshape(-1)

    X = _mask(X.reshape(-1, X.shape[-1]), valid_lens, value=-1e6)
    return torch.nn.functional.softmax(X, dim=-1)


In [7]:
class DotProductAttention():
  def __init__(self):
    pass

  def __repr__(self):
    return f"DotProductAttention()"

  def __call__(self, queries, keys, values, valid_lens=None):
    d = queries.shape[-1]
    scores = torch.bmm(queries, keys.transpose(1,2))/math.sqrt(d)
    weights = masked_softmax(scores, valid_lens)
    return torch.bmm(weights, values)

In [8]:
class AdditiveAttention():
  def __init__(self, key_hidden_state, query_hidden_state, new_hidden_state):
    #shape of keys and values = [seq_len, batch_size, n_hidden]
    #hidden state of decoder(query) = [batch_size, n_hidden]
    self.Wk = Linear(key_hidden_state, new_hidden_state)
    self.Wq = Linear(query_hidden_state, new_hidden_state)
    self.Wv = Linear(new_hidden_state, 1)
    self.tanh = Tanh()

  def __repr__(self):
    return f"AdditiveAttention()"

  def __call__(self, keys, queries, values, valid_lens):
    features = self.Wk(keys) + self.Wq(queries)
    scores = self.tanh(features) @ self.Wv
    scores = scores.permute(1, 2, 0)
    values = values.permute(1, 0, 2)
    self.weights = masked_softmax(scores.squeeze(1), valid_lens)
    return torch.bmm(self.weights, values)

  def parameters(self):
    return [self.Wk, self.Wq, self.Wv]

In [ ]:
class RNNEncoder():
  def __init__(self,
               vocab_size, #number of unique tokens
               feature_size, #number of embedding features
               n_neurons, #number of neurons in the RNN layers
               ):
    self.vocab_size = vocab_size
    self.feature_size = feature_size
    self.neurons = n_neurons

    self.embedding = Embedding(vocab_size, feature_size)

    self.rnn1 = BasicRNN(feature_size, n_neurons)

    #look at hidden state initialization
    self.rnn2 = BasicRNN(n_neurons, n_neurons)


  def __call__(self, x):
    dense_input = self.embedding(x)
    # [batch_size, seq_len, feature_size]
    output1, h1 = self.rnn1(dense_input)
    # [seq_len, batch_size, n_hidden], [batch_size, n_hidden]
    output1 = output1.permute(1, 0, 2)
    # [batch_size, seq_len, n_hidden], [batch_size, n_hidden]
    output, h2 = self.rnn2(output1)
    #[num_of_layers, batch_size, n_hidden]
    return (h1, h2)

  def __repr__(self):
    rep = f"Encoder(\nEmbedding={self.embedding.embedding.shape},\nRNN=({self.feature_size, self.neurons}), \nRNN=({self.neurons, self.neurons})\n)"
    return rep

  def parameters(self):
    params = self.embedding.parameters() + self.rnn1.parameters() + self.rnn2.parameters()
    return params

In [ ]:
class RNNAttentionDecoder():
  def __init__(self, feature_size, vocab_size, n_neurons):
    self.n_neurons = n_neurons
    self.embeddings = Embedding(vocab_size, feature_size)
    self.rnn1 = BasicRNN(feature_size, n_neurons)
    self.rnn2 = BasicRNN(n_neurons, n_neurons)
    self.attention = None
    self

  def __call__(self, x, h_x, x_valid_lens):
    h1, h2 = h_x
    self.attention = AdditiveAttention(
        h1.shape[-1],
        self.n_neurons,
        20
    )
    dense_input = self.embeddings(x)
    output1, h1 = self.rnn1(x, h1)
    output1 = output1.permute(1, 0, 2)
    output2, h2 = self.rnn2(output1, h2)
    context = self.attention(output2, h2, output2, x_valid_lens)


  def __repr__(self):
    pass

  def parameters(self):
    pass
